# Train Test Split

Combine the filtered AAPL news, fractional-price, and technical features into one point-in-time candidate schema, then reuse the immutable holdout boundary established by the original chronological 80/20 split. Missing feature values are preserved for the later `Clean the Data` stage.


## Build the Complete Feature Schema

- Each deduplicated news item is aligned to the first completed dollar-bar timestamp at or after publication, same-timestamp articles are averaged, and fractional-price plus 51 technical indicators are joined at that event start.
- This produces the fixed 53-feature point-in-time schema while deliberately preserving missing values for the later cleaning stage.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve().parents[1]

from src.data_preprocessing.train_test_split import (
    build_event_candidates,
    build_event_feature_schema,
    chronological_train_test_split,
)

period = "2025-01-01_2025-12-31"
market_feature_dir = PROJECT_ROOT / "data/research_data/market/features"
sentiment_path = PROJECT_ROOT / f"data/research_data/alternative/features/aapl_finbert_sentiment_scores_{period}.parquet"
event_dir = PROJECT_ROOT / "data/research_data/events"
candidate_path = event_dir / f"aapl_news_candidate_split_{period}.parquet"
holdout_boundary = pd.Timestamp("2025-10-16 17:14:45.827270Z")

dollar_bars = pd.read_parquet(market_feature_dir / f"aapl_dollar_bar_{period}.parquet").sort_values("end").drop_duplicates("end", keep="last")
fractional = pd.read_parquet(market_feature_dir / f"aapl_dollar_bar_fractional_{period}.parquet").sort_values("end").drop_duplicates("end", keep="last")
technical = pd.read_parquet(market_feature_dir / f"aapl_dollar_bar_technical_{period}.parquet").sort_values("end").drop_duplicates("end", keep="last")
sentiment = pd.read_parquet(sentiment_path).sort_values("created_at", kind="stable")

In [2]:
candidates = build_event_candidates(sentiment, dollar_bars["end"])
candidate_schema = build_event_feature_schema(candidates, fractional, technical)
feature_columns = [column for column in candidate_schema.columns if column not in {"event_start", "symbol"}]

## Create a Test Set

- The timestamp 2025-10-16 17:14:45.827270 UTC is the immutable boundary produced by the original chronological 80/20 split.
- Supplying the timestamp instead of recalculating a fraction keeps later reruns from moving the holdout; rows before it are development and rows at or after it remain sealed holdout.

In [3]:
development, holdout, manifest = chronological_train_test_split(
    candidate_schema,
    holdout_boundary=holdout_boundary,
)
candidate_split = candidate_schema.merge(manifest, on="event_start", how="left", validate="one_to_one")
candidate_split = candidate_split[["event_start", "symbol", "partition", "holdout_boundary", *feature_columns]]

event_dir.mkdir(parents=True, exist_ok=True)
candidate_split.to_parquet(candidate_path, index=False)
print(candidate_path)


/Users/kwonjunhyuk9/Documents/financial-machine-learning/data/research_data/events/aapl_news_candidate_split_2025-01-01_2025-12-31.parquet


In [4]:
partition_summary = pd.DataFrame(
    {
        "events": [len(development), len(holdout)],
        "start": [development["event_start"].min(), holdout["event_start"].min()],
        "end": [development["event_start"].max(), holdout["event_start"].max()],
    },
    index=pd.Index(["development", "holdout"], name="partition"),
)
display(partition_summary)
print(f"holdout_boundary: {holdout_boundary}")

,events,start,end
partition,,,
development,214,2025-01-02 15:32:58.343014+00:00,2025-10-15 13:30:03.030339+00:00
holdout,46,2025-10-16 18:47:49.330442+00:00,2025-12-18 14:30:00.344027+00:00


holdout_boundary: 2025-10-16 17:14:45.827270+00:00
